# CrediScope Analysis

**Clear decisions. Clear reasons.**

This notebook explores the CreditSense dataset and prepares the data for machine learning.

The aim is to understand the structure of the dataset, identify any issues that need preprocessing, and prepare the features before training a credit-risk model.

In [92]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

In [93]:
df = pd.read_csv("../data/creditsense_applications.csv")
dictionary = pd.read_csv("../data/creditsense_data_dictionary.csv")

In [94]:
df.shape

(6000, 27)

In [95]:
df.head()

,application_id,age_years,sex,marital_status,residency_status,num_dependents,region,employment_status,months_in_current_job,annual_income_gbp,...,has_savings_account,savings_balance_gbp,checking_account_status,loan_amount_gbp,loan_term_months,loan_purpose,has_guarantor,previous_application_declined,application_channel,default_within_24m
0,APP-000001,34,Female,Married,Settled Status,1,North West,Permanent,19,66550,...,Yes,380.0,500 GBP or more,13100,36,Home improvement,No,No,Online,0
1,APP-000002,43,Female,Married,Settled Status,1,South East,Permanent,29,20820,...,No,0.0,Overdrawn,9700,60,Medical,No,No,Branch,1
2,APP-000003,47,Male,Single,UK Citizen,0,South East,Permanent,115,32710,...,No,0.0,500 GBP or more,8600,36,Home improvement,No,No,Online,0
3,APP-000004,63,Male,Married,UK Citizen,1,London,Permanent,146,47300,...,Yes,NaN,0 to 500 GBP,25600,36,Home improvement,No,No,Online,0
4,APP-000005,40,Male,Single,UK Citizen,1,Yorkshire,Permanent,22,41400,...,Yes,860.0,0 to 500 GBP,3500,36,Car,Yes,No,Branch,0


In [96]:
df.columns.tolist()

['application_id',
 'age_years',
 'sex',
 'marital_status',
 'residency_status',
 'num_dependents',
 'region',
 'employment_status',
 'months_in_current_job',
 'annual_income_gbp',
 'monthly_expenses_gbp',
 'housing_status',
 'years_at_current_address',
 'existing_debt_gbp',
 'num_existing_credit_lines',
 'num_late_payments_24m',
 'credit_history',
 'has_savings_account',
 'savings_balance_gbp',
 'checking_account_status',
 'loan_amount_gbp',
 'loan_term_months',
 'loan_purpose',
 'has_guarantor',
 'previous_application_declined',
 'application_channel',
 'default_within_24m']

In [97]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   application_id                 6000 non-null   str    
 1   age_years                      6000 non-null   int64  
 2   sex                            6000 non-null   str    
 3   marital_status                 6000 non-null   str    
 4   residency_status               6000 non-null   str    
 5   num_dependents                 6000 non-null   int64  
 6   region                         6000 non-null   str    
 7   employment_status              6000 non-null   str    
 8   months_in_current_job          6000 non-null   int64  
 9   annual_income_gbp              6000 non-null   int64  
 10  monthly_expenses_gbp           6000 non-null   int64  
 11  housing_status                 6000 non-null   str    
 12  years_at_current_address       5832 non-null   float64
 13 

In [98]:
df.isna().sum().sort_values(ascending=False)

savings_balance_gbp              258
years_at_current_address         168
application_id                     0
marital_status                     0
residency_status                   0
num_dependents                     0
region                             0
employment_status                  0
months_in_current_job              0
age_years                          0
sex                                0
monthly_expenses_gbp               0
annual_income_gbp                  0
existing_debt_gbp                  0
housing_status                     0
num_existing_credit_lines          0
num_late_payments_24m              0
credit_history                     0
has_savings_account                0
checking_account_status            0
loan_amount_gbp                    0
loan_term_months                   0
loan_purpose                       0
has_guarantor                      0
previous_application_declined      0
application_channel                0
default_within_24m                 0
d

## Missing Values

Only two features contain missing values:

- `savings_balance_gbp` has 258 missing values.
- `years_at_current_address` has 168 missing values.

All other columns are complete.

These missing values will be investigated before deciding how they should be handled during preprocessing.

In [99]:
df["default_within_24m"].value_counts()

default_within_24m
0    4467
1    1533
Name: count, dtype: int64

In [100]:
df["default_within_24m"].value_counts(normalize=True) * 100

default_within_24m
0    74.45
1    25.55
Name: proportion, dtype: float64

## Target Variable

The target variable is `default_within_24m`.

- `0` means the applicant did not default within 24 months.
- `1` means the applicant did default within 24 months.

The target is moderately imbalanced, with 25.55% of applicants in the default class.

In [101]:
dictionary.head(30)

,column,type,description,protected_attribute,notes
0,application_id,string,Unique application reference,No,Identifier only - must NOT be used as a feature
1,age_years,integer,Applicant age at application,Yes,Protected characteristic under Equality Act 2010
2,sex,categorical,Female / Male,Yes,Protected characteristic under Equality Act 2010
3,marital_status,categorical,Single / Married / Divorced / Widowed,Yes,Correlates strongly with age
4,residency_status,categorical,UK Citizen / Settled Status / Visa Holder,Yes,Proxy for nationality - protected characteristic
5,num_dependents,integer,Number of financial dependents,No,Correlates with marital_status
6,region,categorical,UK region of residence,No,Can act as a proxy for other attributes
7,employment_status,categorical,Permanent / Fixed-term / Self-employed / Zero-...,No,Retired appears only in older applicants
8,months_in_current_job,integer,Tenure in current employment (months),No,Zero for unemployed applicants - structural ze...
9,annual_income_gbp,integer,Gross annual income in GBP,No,Not adjusted for region


## Initial Data Dictionary Observations

The data dictionary highlights several features that will need careful handling during preprocessing.

- `application_id` is an identifier and should not be used as a predictive feature.
- `age_years`, `sex`, `marital_status`, `residency_status` and `housing_status` are marked as protected attributes.
- `region` is not marked as protected, but it may act as a proxy for other characteristics.
- `credit_history` is ordinal.
- `checking_account_status` is ordinal, but the `No account` category does not fit naturally on the same ordered scale.
- `months_in_current_job = 0` for unemployed applicants is a structural zero and represents valid information.
- Missing values occur in `years_at_current_address` and `savings_balance_gbp`.

In [102]:
numerical_columns = df.select_dtypes(
   include=["int64", "float64"] 
).columns.tolist()

categorical_columns = df.select_dtypes(
   include="str" 
).columns.tolist()

In [103]:
print("Numerical Columns:")
print(numerical_columns)

print("\nCategorical Columns:")
print(categorical_columns)


Numerical Columns:
['age_years', 'num_dependents', 'months_in_current_job', 'annual_income_gbp', 'monthly_expenses_gbp', 'years_at_current_address', 'existing_debt_gbp', 'num_existing_credit_lines', 'num_late_payments_24m', 'savings_balance_gbp', 'loan_amount_gbp', 'loan_term_months', 'default_within_24m']

Categorical Columns:
['application_id', 'sex', 'marital_status', 'residency_status', 'region', 'employment_status', 'housing_status', 'credit_history', 'has_savings_account', 'checking_account_status', 'loan_purpose', 'has_guarantor', 'previous_application_declined', 'application_channel']


In [104]:
target_column = "default_within_24m"
id_column = "application_id"

numerical_features = [
    col for col in numerical_columns 
    if col not in [target_column, id_column]
]

categorical_features = [
    col for col in categorical_columns
    if col != id_column
]

In [105]:
print("Numerical Columns:") 
print(numerical_features)

print("\nCategorical Columns:")
print(categorical_features)

Numerical Columns:
['age_years', 'num_dependents', 'months_in_current_job', 'annual_income_gbp', 'monthly_expenses_gbp', 'years_at_current_address', 'existing_debt_gbp', 'num_existing_credit_lines', 'num_late_payments_24m', 'savings_balance_gbp', 'loan_amount_gbp', 'loan_term_months']

Categorical Columns:
['sex', 'marital_status', 'residency_status', 'region', 'employment_status', 'housing_status', 'credit_history', 'has_savings_account', 'checking_account_status', 'loan_purpose', 'has_guarantor', 'previous_application_declined', 'application_channel']


## Feature Groups

The dataset contains 25 predictive features.

- 12 numerical features
- 13 categorical features
- `application_id` is excluded because it is only an identifier
- `default_within_24m` is the target variable

# Data Preprocessing

This section prepares the dataset for machine learning.

A separate working copy of the original dataframe will be used so that the raw data remains unchanged while preprocessing decisions are applied.

In [106]:
df_clean = df.copy()

In [107]:
print("Original dataset shape:", df.shape)
print("Working copy shape:", df_clean.shape)

Original dataset shape: (6000, 27)
Working copy shape: (6000, 27)


In [108]:
df_clean.isna().sum()[df_clean.isna().sum() >0]

years_at_current_address    168
savings_balance_gbp         258
dtype: int64

## Missing Value Investigation

Two features contain missing values:

- `years_at_current_address`
- `savings_balance_gbp`

Before filling these values, their distributions and meanings will be checked to choose an appropriate preprocessing method.

In [109]:
df_clean["years_at_current_address"].describe()

count    5832.000000
mean        5.169513
std         4.893061
min         0.000000
25%         1.900000
50%         3.700000
75%         6.800000
max        40.000000
Name: years_at_current_address, dtype: float64

In [110]:
df_clean["savings_balance_gbp"].describe()

count      5742.000000
mean       3206.849530
std        7875.004884
min           0.000000
25%           0.000000
50%         610.000000
75%        3250.000000
max      160000.000000
Name: savings_balance_gbp, dtype: float64

In [111]:
df_clean.groupby("has_savings_account")["savings_balance_gbp"].apply(
    lambda x: x.isna().sum()
)

has_savings_account
No       0
Yes    258
Name: savings_balance_gbp, dtype: int64

## Missing Value Decision

The two features with missing values are `years_at_current_address` and `savings_balance_gbp`.

Both features are positively skewed, so the median is preferred to the mean because it is less affected by large values.

For `savings_balance_gbp`, all missing values occur among applicants who do have a savings account. Applicants without a savings account already have a meaningful balance of £0, so those zeros must be preserved.

In [112]:
df_clean["years_at_current_address"] = (
    df_clean["years_at_current_address"]
    .fillna(df_clean["years_at_current_address"].median())
)

df_clean["savings_balance_gbp"] = (
    df_clean["savings_balance_gbp"]
    .fillna(df_clean["savings_balance_gbp"].median())
)

In [113]:
df_clean[
    ["years_at_current_address", "savings_balance_gbp"]
].isna().sum()

years_at_current_address    0
savings_balance_gbp         0
dtype: int64

In [114]:
df_clean.isna().sum().sum()

np.int64(0)

### Missing Value Result

Median imputation was applied to the two features containing missing values.

After preprocessing, the working dataset contains no missing values.

In [115]:
df_clean.groupby("employment_status")["months_in_current_job"].describe()

,count,mean,std,min,25%,50%,75%,max
employment_status,,,,,,,,
Fixed-term,606.0,21.016502,13.643140,2.0,11.0,17.0,29.0,89.0
Permanent,4125.0,53.890667,34.846809,3.0,28.0,46.0,73.0,245.0
Retired,77.0,111.389610,44.850382,41.0,74.0,109.0,136.0,242.0
Self-employed,679.0,55.587629,37.494002,3.0,30.0,46.0,72.0,250.0
Unemployed,227.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
Zero-hours,286.0,21.363636,13.552003,1.0,11.0,19.0,30.0,83.0


## Structural Zero Check

The `months_in_current_job` feature contains zero values for unemployed applicants.

All 227 unemployed applicants have `months_in_current_job = 0`, which indicates that these values are structural zeros rather than missing data.

These values will therefore be preserved.

In [116]:
binary_columns = [
    "has_savings_account",
    "has_guarantor",
    "previous_application_declined",
]

In [117]:
for col in binary_columns:
    df_clean[col] = df_clean[col].map({"Yes": 1, "No": 0})

In [118]:
for col in binary_columns:
    print(col, df_clean[col].value_counts())

has_savings_account has_savings_account
1    3495
0    2505
Name: count, dtype: int64
has_guarantor has_guarantor
0    5432
1     568
Name: count, dtype: int64
previous_application_declined previous_application_declined
0    5251
1     749
Name: count, dtype: int64


## Binary Feature Encoding

Three Yes/No categorical features were converted to numerical binary values:

- `has_savings_account`
- `has_guarantor`
- `previous_application_declined`

`Yes` was encoded as `1` and `No` as `0`.

In [119]:
df_clean["credit_history"].value_counts()

credit_history
Existing credits paid    1935
All paid on time         1727
Past delays              1356
No credits taken          882
Critical account          100
Name: count, dtype: int64

## Ordinal Encoding

The `credit_history` feature has categories that represent different levels of credit risk.

An ordinal encoding will be used so that lower values represent better credit history and higher values represent greater risk.

Proposed order:

1. `All paid on time`
2. `Existing credits paid`
3. `No credits taken`
4. `Past delays`
5. `Critical account`

In [120]:
credit_history_mapping = {
    "All paid on time": 0,
    "Existing credits paid": 1,
    "No credits taken": 2,
    "Past delays": 3,
    "Critical account": 4,
}

In [121]:
df_clean["credit_history"].value_counts().sort_index()

credit_history
All paid on time         1727
Critical account          100
Existing credits paid    1935
No credits taken          882
Past delays              1356
Name: count, dtype: int64

In [122]:
df_clean["credit_history"].isna().sum()

np.int64(0)

In [123]:
df_clean["checking_account_status"].value_counts()

checking_account_status
0 to 500 GBP       2185
500 GBP or more    1971
Overdrawn          1252
No account          592
Name: count, dtype: int64

### Checking Account Status

The `checking_account_status` feature contains three naturally ordered categories:

- `Overdrawn`
- `0 to 500 GBP`
- `500 GBP or more`

The category `No account` does not fit naturally on the same ordinal scale.

To preserve this distinction, `No account` will be handled separately rather than being forced into the ordered ranking.

In [124]:
df_clean["has_checking_account"] = (
    df_clean["checking_account_status"] != "No account"
).astype(int)

In [125]:
df_clean["has_checking_account"].value_counts()

has_checking_account
1    5408
0     592
Name: count, dtype: int64

In [126]:
checking_status_mapping = {
    "Overdrawn": 0,
    "0 to 500 GBP": 1,
    "500 GBP or more": 2,
}

In [127]:
df_clean["checking_account_status"] = (
    df_clean["checking_account_status"]
    .map(checking_status_mapping)
)

In [128]:
df_clean["checking_account_status"].value_counts(dropna=False).sort_index()

checking_account_status
0.0    1252
1.0    2185
2.0    1971
NaN     592
Name: count, dtype: int64

In [129]:
df_clean["checking_account_status"] = (
    df_clean["checking_account_status"]
    .fillna(0)
    .astype(int)
)

In [130]:
df_clean[
    ["checking_account_status", "has_checking_account"]
].value_counts().sort_index()

checking_account_status  has_checking_account
0                        0                        592
                         1                       1252
1                        1                       2185
2                        1                       1971
Name: count, dtype: int64

### Checking Account Encoding Result

The ordered checking account categories were encoded numerically, while `No account` was separated using the `has_checking_account` indicator.

This preserves the distinction between:

- applicants with no checking account
- applicants who are overdrawn
- applicants with lower balances
- applicants with higher balances

In [131]:
remaining_categorical = df_clean.select_dtypes(include="object").columns.tolist()

remaining_categorical

C:\Users\Richa\AppData\Local\Temp\ipykernel_2800\448189261.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  remaining_categorical = df_clean.select_dtypes(include="object").columns.tolist()


['application_id',
 'sex',
 'marital_status',
 'residency_status',
 'region',
 'employment_status',
 'housing_status',
 'credit_history',
 'loan_purpose',
 'application_channel']

In [132]:
df_clean["credit_history"].head(10)

0    Existing credits paid
1         All paid on time
2         No credits taken
3    Existing credits paid
4              Past delays
5    Existing credits paid
6              Past delays
7              Past delays
8         All paid on time
9    Existing credits paid
Name: credit_history, dtype: str

In [133]:
df_clean["credit_history"] = df_clean["credit_history"].map(
    credit_history_mapping
)

In [134]:
df_clean["credit_history"].value_counts().sort_index()

credit_history
0    1727
1    1935
2     882
3    1356
4     100
Name: count, dtype: int64

In [135]:
df_clean["credit_history"].isna().sum()

np.int64(0)

In [136]:
remaining_categorical = df_clean.select_dtypes(include="str").columns.tolist()

remaining_categorical

['application_id',
 'sex',
 'marital_status',
 'residency_status',
 'region',
 'employment_status',
 'housing_status',
 'loan_purpose',
 'application_channel']

In [137]:
nominal_features = [
    "region",
    "employment_status",
    "loan_purpose",
    "application_channel",
]

## Nominal Categorical Features

The following features have categories with no natural ranking:

- `region`
- `employment_status`
- `loan_purpose`
- `application_channel`

These features will be prepared using one-hot encoding.

Other categorical features such as `sex`, `marital_status`, `residency_status` and `housing_status` require a separate modelling decision because of their protected or sensitive nature.

## Protected and Sensitive Attributes

Several categorical features are identified as protected or sensitive within the data dictionary.

These features will not be used directly as predictive inputs in the initial model:

- `sex`
- `marital_status`
- `residency_status`
- `housing_status`

They will be retained in the dataset for later fairness analysis.

`region` is not formally marked as protected, but it may act as a proxy for other characteristics, so its use will be considered carefully.

In [138]:
protected_features = [
    "sex",
    "marital_status",
    "residency_status",
    "housing_status"
]

In [139]:
excluded_features = [
    "application_id",
    "default_within_24m",
] + protected_features

In [140]:
model_features = [
    col for col in df_clean.columns 
    if col not in excluded_features
]

In [141]:
model_features

['age_years',
 'num_dependents',
 'region',
 'employment_status',
 'months_in_current_job',
 'annual_income_gbp',
 'monthly_expenses_gbp',
 'years_at_current_address',
 'existing_debt_gbp',
 'num_existing_credit_lines',
 'num_late_payments_24m',
 'credit_history',
 'has_savings_account',
 'savings_balance_gbp',
 'checking_account_status',
 'loan_amount_gbp',
 'loan_term_months',
 'loan_purpose',
 'has_guarantor',
 'previous_application_declined',
 'application_channel',
 'has_checking_account']

In [142]:
protected_features = [
    "age_years",
    "sex",
    "marital_status",
    "residency_status",
    "housing_status"
]

In [143]:
excluded_features = [
    "application_id",
    "default_within_24m"
] + protected_features

In [144]:
model_features = [
    col for col in df_clean.columns
    if col not in excluded_features
]

In [145]:
model_features

['num_dependents',
 'region',
 'employment_status',
 'months_in_current_job',
 'annual_income_gbp',
 'monthly_expenses_gbp',
 'years_at_current_address',
 'existing_debt_gbp',
 'num_existing_credit_lines',
 'num_late_payments_24m',
 'credit_history',
 'has_savings_account',
 'savings_balance_gbp',
 'checking_account_status',
 'loan_amount_gbp',
 'loan_term_months',
 'loan_purpose',
 'has_guarantor',
 'previous_application_declined',
 'application_channel',
 'has_checking_account']

## Initial Model Feature Set

The initial model excludes:

- `application_id`
- the target variable `default_within_24m`
- protected attributes identified in the data dictionary

The remaining features form the initial modelling feature set.

`region` is retained at this stage, but it may act as a proxy for protected characteristics and will be reviewed during fairness analysis.

In [147]:
nominal_features

['region', 'employment_status', 'loan_purpose', 'application_channel']

In [148]:
df_clean = pd.get_dummies(
    df_clean,
    columns=nominal_features,
    drop_first=True,
    dtype=int
)

In [149]:
df_clean.shape

(6000, 48)

In [150]:
[col for col in df_clean.columns if col.startswith("region_")][:10]

['region_Midlands',
 'region_North East',
 'region_North West',
 'region_Northern Ireland',
 'region_Scotland',
 'region_South East',
 'region_South West',
 'region_Wales',
 'region_Yorkshire']

## One-Hot Encoding

The nominal categorical features were converted into binary indicator columns using one-hot encoding.

The encoded features were:

- `region`
- `employment_status`
- `loan_purpose`
- `application_channel`

`drop_first=True` was used so that one category from each feature acts as a reference category, reducing redundant information.

The dataset increased from 27 columns to 48 columns while retaining all 6,000 application records.

In [152]:
df_clean.select_dtypes(include="object").columns.tolist()

C:\Users\Richa\AppData\Local\Temp\ipykernel_2800\2435950628.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_clean.select_dtypes(include="object").columns.tolist()


['application_id',
 'sex',
 'marital_status',
 'residency_status',
 'housing_status']

### Encoding Check

After one-hot encoding, the only remaining text columns are the application identifier and protected/sensitive attributes.

These columns are not included directly in the initial predictive model.

In [154]:
df_clean["debt_to_income_ratio"] = (
    df_clean["existing_debt_gbp"] /
    df_clean["annual_income_gbp"]
)

In [155]:
df_clean["debt_to_income_ratio"].describe()

count    6000.000000
mean        0.307106
std         0.284312
min         0.000000
25%         0.000000
50%         0.261718
75%         0.560400
max         0.849748
Name: debt_to_income_ratio, dtype: float64

## Feature Engineering

A `debt_to_income_ratio` feature was created by dividing existing debt by annual income.

This provides a relative measure of debt burden, which may be more informative than considering debt or income separately.

In [156]:
df_clean["expenses_to_income_ratio"] = (
    df_clean["monthly_expenses_gbp"] * 12 /
    df_clean["annual_income_gbp"]
)    

In [157]:
df_clean["expenses_to_income_ratio"].describe()

count    6000.000000
mean        0.788505
std         0.359927
min         0.340003
25%         0.566188
50%         0.704567
75%         0.921250
max         4.596923
Name: expenses_to_income_ratio, dtype: float64

In [158]:
df_clean = df_clean.rename(
    columns={"expenses_to_income_ratio": "expense_to_income_ratio"}
)

In [159]:
"expense_to_income_ratio" in df_clean.columns

True

In [160]:
df_clean[df_clean["expense_to_income_ratio"] > 1][
    [
        "annual_income_gbp",
        "monthly_expenses_gbp",
        "expense_to_income_ratio"
    ]
].head(10)    

,annual_income_gbp,monthly_expenses_gbp,expense_to_income_ratio
1,20820,2460,1.417867
16,29280,3140,1.286885
32,17950,1690,1.129805
33,29240,2460,1.009576
52,12560,1330,1.270701
59,16610,2160,1.560506
61,16160,1560,1.158416
68,31540,2700,1.027267
73,25960,2390,1.104777
74,12580,1650,1.573927


In [161]:
(df_clean["expense_to_income_ratio"] > 1).sum()

np.int64(1123)

### Expense-to-Income Ratio

An `expense_to_income_ratio` feature was created by annualising monthly expenses and dividing them by annual income.

Some applicants have a ratio greater than 1, meaning their stated annual expenses exceed their annual income.

This occurs for 1,123 applicants, so it is treated as a meaningful characteristic of the dataset rather than an isolated data error.

## Train/Test Split

The dataset will now be separated into training and testing sets.

The training data will be used to fit the machine-learning model, while the test data will be kept separate so that model performance can be evaluated on unseen applications.

In [162]:
from sklearn.model_selection import train_test_split

In [171]:
X = df_clean.drop(
    columns=[
        "application_id",
        "default_within_24m",
        "age_years",
        "sex",
        "marital_status",
        "residency_status",
        "housing_status"
    ]
)

y = df_clean["default_within_24m"]

In [172]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (6000, 43)
y shape: (6000,)


In [173]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [174]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (4800, 43)
X_test: (1200, 43)
y_train: (4800,)
y_test: (1200,)


In [175]:
print("Training target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training target distribution:
default_within_24m
0    74.458333
1    25.541667
Name: proportion, dtype: float64

Testing target distribution:
default_within_24m
0    74.416667
1    25.583333
Name: proportion, dtype: float64


### Split Verification

A stratified 80/20 train-test split was used.

The class distribution was preserved across both datasets, with approximately 74% non-default cases and 26% default cases in both the training and testing sets.

This helps ensure that model evaluation is performed on a representative test set.

In [176]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [177]:
baseline_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

In [178]:
baseline_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](43,)","['num_dependents','months_in_current_job','annual_income_gbp',..., 'application_channel_Phone','debt_to_income_ratio', 'expense_to_income_ratio']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,43
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [179]:
y_pred = baseline_model.predict(X_test)

In [180]:
from sklearn.metrics import accuracy_score

In [181]:
accurancy = accuracy_score(y_test, y_pred)

print("Baseline Model Accuracy:", accurancy)

Baseline Model Accuracy: 0.8


In [182]:
from sklearn.metrics import classification_report

In [183]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.92      0.87       893
           1       0.65      0.46      0.54       307

    accuracy                           0.80      1200
   macro avg       0.74      0.69      0.71      1200
weighted avg       0.79      0.80      0.79      1200



### Baseline Classification Performance

The baseline Logistic Regression model achieved an accuracy of 80%.

However, performance differed between the two classes.

For applicants who defaulted (`class 1`), the model achieved:

- Precision: 0.65
- Recall: 0.46
- F1-score: 0.54

The relatively low recall means the model fails to identify more than half of the applicants who later default.

This shows why accuracy alone is not sufficient for evaluating the model.


In [184]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)

array([[818,  75],
       [165, 142]])

### Confusion Matrix Interpretation

The confusion matrix for the baseline Logistic Regression model produced:

- 818 true negatives
- 75 false positives
- 165 false negatives
- 142 true positives

The most significant weakness is the 165 false negatives. These are applicants who actually defaulted but were predicted as non-default.

This supports the earlier finding that the baseline model has relatively low recall for the default class and may fail to identify a substantial number of higher-risk applicants.

In [187]:
from sklearn.metrics import roc_auc_score

In [188]:
y_prob = baseline_model.predict_proba(X_test)[:, 1]

In [189]:
roc_auc = roc_auc_score(y_test, y_prob)

print("Baseline ROC AUC Score:", roc_auc)

Baseline ROC AUC Score: 0.8111259853146624


### ROC-AUC

The baseline Logistic Regression model achieved a ROC-AUC score of approximately 0.81.

This indicates that the model has a reasonably strong ability to distinguish between applicants who default and those who do not.

However, this does not remove the concern identified in the confusion matrix, where the model missed a substantial number of actual default cases.